# 데이터셋 탐색(EDA)  –  Hotel Recommender

전처리된 두 데이터셋을 탐색하고, 추천 모델 학습·테스트에 쓰기 전에 알아야 할 특성을 정리합니다.

| 데이터셋 | 역할 | 파일 |
|---|---|---|
| HotelRec (TripAdvisor) | 모델 설계·학습 | `hotelrec_train.csv`, `hotelrec_partial.csv` |
| 515K Hotel Reviews in Europe (Booking.com) | 일반화 테스트 | `kaggle_test.csv`, `hotel_meta_kaggle.csv` |

**구성**: A. HotelRec  /  B. Kaggle  /  C. 두 데이터셋 비교  /  D. 전처리 전후 비교  /  E. 핵심 수치와 해석 메모

**실행 방법**: 아래 설정 셀의 경로만 수정한 뒤 `Run All`. 그래프는 `eda_figures/` 폴더에 PNG로 저장됩니다.
필요 패키지: `pandas`, `numpy`, `matplotlib` (`pip install pandas numpy matplotlib`)

> HotelRec 파일(리뷰 약 120만 건, 본문 포함)을 읽으므로 메모리를 2~3GB 정도 사용하고 1~3분 걸립니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

# ================= 경로 설정 (본인 환경에 맞게 수정) =================
HOTELREC_DIR = Path(r"D:/hotelrec")
KAGGLE_DIR   = Path(r"../data/processing/kaggle")
FIG_DIR      = Path("eda_figures")
# =====================================================================
FIG_DIR.mkdir(exist_ok=True)

# 한글 폰트 (설치된 것 중 첫 번째 사용)
_installed = {f.name for f in font_manager.fontManager.ttflist}
for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if _f in _installed:
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.options.display.float_format = "{:,.3f}".format
pd.options.display.max_columns = 50

ASPECTS = ["service", "cleanliness", "value", "rooms", "location", "sleep_quality"]
C_TRAIN, C_PART = "#1f77b4", "#ff7f0e"

def save(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

---
# A. HotelRec (학습용)

`hotelrec_train.csv`는 세부 평점 6개(service, cleanliness, value, rooms, location, sleep quality)가 **모두 있는** 리뷰,
`hotelrec_partial.csv`는 일부 또는 전부 **없는** 리뷰입니다. 두 파일은 같은 `user_id`/`hotel_id` 체계를 씁니다.

In [ ]:
def load_reviews(path):
    """본문은 길이(단어 수)만 계산하고 버려서 메모리를 아낀다."""
    parts = []
    for ch in pd.read_csv(path, chunksize=200_000):
        ch["title_words"] = ch["title"].fillna("").str.split().str.len()
        ch["text_words"] = ch["text"].fillna("").str.split().str.len()
        parts.append(ch.drop(columns=["title", "text"]))
    return pd.concat(parts, ignore_index=True)

train = load_reviews(HOTELREC_DIR / "hotelrec_train.csv")
partial = load_reviews(HOTELREC_DIR / "hotelrec_partial.csv")
train["group"], partial["group"] = "train", "partial"
hr = pd.concat([train, partial], ignore_index=True)
hr["date"] = pd.to_datetime(hr["date"])
hr["year"] = hr["date"].dt.year
hr["n_aspects"] = hr[ASPECTS].notna().sum(axis=1)
hotel_map = pd.read_csv(HOTELREC_DIR / "hotel_map.csv")
print(f"불러오기 완료: train {len(train):,}행, partial {len(partial):,}행")

## A-1. 기본 정보 (행 수, 컬럼, 데이터 타입, 결측, 중복)

In [ ]:
overview = pd.DataFrame({
    "행 수": [len(train), len(partial), len(hr)],
    "사용자 수": [train.user_id.nunique(), partial.user_id.nunique(), hr.user_id.nunique()],
    "호텔 수": [train.hotel_id.nunique(), partial.hotel_id.nunique(), hr.hotel_id.nunique()],
}, index=["train", "partial", "합계"])
display(overview)

print("컬럼과 데이터 타입")
display(hr.drop(columns=["group", "year", "n_aspects"]).dtypes.rename("dtype").to_frame().T)

print("결측 비율 (%)")
miss = pd.DataFrame({
    "train": train[["rating", *ASPECTS]].isna().mean() * 100,
    "partial": partial[["rating", *ASPECTS]].isna().mean() * 100,
    "합계": hr[["rating", *ASPECTS]].isna().mean() * 100,
})
display(miss)

dup = hr.duplicated(["user_id", "hotel_id"]).sum()
print(f"(사용자, 호텔) 중복: {dup}건 | 본문이 비어 있는 리뷰 비율: {(hr.text_words == 0).mean():.2%}")
display(hr[["rating", *ASPECTS, "title_words", "text_words"]].describe().T)

## A-2. 세부 평점 보유 현황 (train / partial 분리 근거)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

cnt = hr["n_aspects"].value_counts().sort_index()
ax[0].bar(cnt.index.astype(str), cnt.values / len(hr) * 100, color=[C_PART] * 6 + [C_TRAIN])
ax[0].set_title("리뷰가 가진 세부 평점 개수 분포")
ax[0].set_xlabel("세부 평점 개수 (0~6)")
ax[0].set_ylabel("리뷰 비율 (%)")

miss_rate = hr[ASPECTS].isna().mean() * 100
ax[1].barh(miss_rate.index[::-1], miss_rate.values[::-1], color="#888")
ax[1].set_title("세부 평점별 결측 비율 (train+partial)")
ax[1].set_xlabel("결측 비율 (%)")
for i, v in enumerate(miss_rate.values[::-1]):
    ax[1].text(v + 0.5, i, f"{v:.1f}%", va="center")
save("A2_aspect_availability")

print(f"세부 평점 6개 모두 있는 리뷰: {(hr.n_aspects == 6).mean():.1%}")

## A-3. 평점 분포

In [ ]:
def rating_dist(df):
    r = df["rating"].round().astype(int).clip(1, 5)
    return r.value_counts(normalize=True).reindex(range(1, 6), fill_value=0) * 100

d_train, d_part = rating_dist(train), rating_dist(partial)
x = np.arange(1, 6)
plt.figure(figsize=(7, 4))
plt.bar(x - 0.2, d_train.values, 0.4, label="train", color=C_TRAIN)
plt.bar(x + 0.2, d_part.values, 0.4, label="partial", color=C_PART)
plt.xlabel("전체 평점 (rating)")
plt.ylabel("비율 (%)")
plt.title("전체 평점 분포: train vs partial")
plt.legend()
save("A3_rating_distribution")

print(f"평균 평점  train {train.rating.mean():.3f} / partial {partial.rating.mean():.3f} / 합계 {hr.rating.mean():.3f}")
print("train과 partial의 평점 분포가 크게 다르면, 세부 평점이 있는 리뷰만 학습에 쓸 때 편향이 생길 수 있음")

In [ ]:
# 세부 평점별 분포 (train: 6개 모두 있는 리뷰)
fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharey=True)
for ax, c in zip(axes.ravel(), ASPECTS):
    vc = train[c].round().astype(int).clip(1, 5).value_counts(normalize=True).reindex(range(1, 6), fill_value=0) * 100
    ax.bar(vc.index, vc.values, color=C_TRAIN)
    ax.set_title(f"{c} (평균 {train[c].mean():.2f})")
    ax.set_xlabel("점수")
axes[0, 0].set_ylabel("비율 (%)")
axes[1, 0].set_ylabel("비율 (%)")
plt.suptitle("세부 평점별 분포 (train)", y=1.02)
save("A3_aspect_distributions")

In [ ]:
# 세부 평점 간 상관관계 (train)
corr = train[["rating", *ASPECTS]].corr()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr.values, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
ax.grid(False)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                color="white" if corr.values[i, j] > 0.6 else "black")
plt.colorbar(im)
ax.set_title("전체 평점과 세부 평점의 상관계수 (train)")
save("A3_aspect_correlation")

print("전체 평점과의 상관이 큰 순서:")
display(corr["rating"].drop("rating").sort_values(ascending=False).to_frame("corr with rating"))

## A-4. 사용자·호텔별 리뷰 수 (희소도)

협업 필터링과 사용자별 aspect 중요도 회귀의 가능 범위를 가늠하는 핵심 지표입니다.

In [ ]:
u_all = hr.groupby("user_id").size()
u_train = train.groupby("user_id").size()
h_all = hr.groupby("hotel_id").size()

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(u_all.clip(upper=40), bins=range(1, 42), color="#555")
ax[0].set_yscale("log")
ax[0].set_title("사용자당 리뷰 수 (전체, 40건 초과는 40으로 표시)")
ax[0].set_xlabel("리뷰 수")
ax[0].set_ylabel("사용자 수 (log)")

ax[1].hist(u_train.clip(upper=40), bins=range(1, 42), color=C_TRAIN)
ax[1].set_yscale("log")
ax[1].set_title("사용자당 '세부 평점 완전' 리뷰 수 (train)")
ax[1].set_xlabel("리뷰 수")

ax[2].hist(np.log10(h_all), bins=40, color="#555")
ax[2].set_title("호텔당 리뷰 수 (전체, log10)")
ax[2].set_xlabel("log10(리뷰 수)")
save("A4_user_hotel_counts")

q = [0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
tbl = pd.DataFrame({"사용자당(전체)": u_all.quantile(q), "사용자당(train)": u_train.quantile(q),
                    "호텔당(전체)": h_all.quantile(q)})
tbl.index = [f"{int(i * 100)}%" for i in q]
display(tbl)

n_users, n_hotels = hr.user_id.nunique(), hr.hotel_id.nunique()
print(f"사용자 {n_users:,} x 호텔 {n_hotels:,}, 리뷰 {len(hr):,} -> 밀도 {len(hr) / (n_users * n_hotels):.4%} (희소도 {1 - len(hr) / (n_users * n_hotels):.4%})")

In [ ]:
# train에서 사용자당 완전한 리뷰가 N건 이상인 사용자 수
th = [1, 3, 5, 7, 10, 15]
res = pd.DataFrame({"사용자 수": [int((u_train >= n).sum()) for n in th],
                    "비율(%)": [(u_train >= n).mean() * 100 for n in th]},
                   index=[f"{n}건 이상" for n in th])
display(res)

plt.figure(figsize=(6, 4))
plt.bar(res.index, res["사용자 수"], color=C_TRAIN)
for i, v in enumerate(res["사용자 수"]):
    plt.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.title("세부 평점 완전 리뷰가 N건 이상인 사용자 수 (train)")
plt.ylabel("사용자 수")
plt.xticks(rotation=20)
save("A4_train_user_thresholds")
print("사용자별 aspect 중요도 회귀(변수 6개+절편)는 사용자당 7건 이상이면 안정적으로 추정 가능 -> 해당 사용자 수를 위 표에서 확인")

## A-5. 시간 분포

In [ ]:
by_year = hr.groupby("year").size()
comp = hr.groupby("year")["n_aspects"].apply(lambda s: (s == 6).mean() * 100)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(by_year.index, by_year.values, color="#555")
ax[0].set_title("연도별 리뷰 수")
ax[0].set_xlabel("연도")
ax[1].plot(comp.index, comp.values, marker="o", color=C_TRAIN)
ax[1].set_title("연도별 '세부 평점 6개 완전' 리뷰 비율 (%)")
ax[1].set_xlabel("연도")
ax[1].set_ylim(0, 100)
save("A5_time_distribution")
print(f"리뷰 기간: {hr.date.min().date()} ~ {hr.date.max().date()}")

## A-6. 리뷰 텍스트 길이

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(hr.text_words.clip(upper=500), bins=50, color="#555")
ax[0].set_title("본문 단어 수 (500 초과는 500으로 표시)")
ax[0].set_xlabel("단어 수")
ax[1].hist(hr.title_words.clip(upper=30), bins=30, color="#555")
ax[1].set_title("제목 단어 수 (30 초과는 30으로 표시)")
ax[1].set_xlabel("단어 수")
save("A6_text_length")
display(hr[["text_words", "title_words"]].describe(percentiles=[.25, .5, .75, .9, .99]).T)
print("콘텐츠 기반(TF-IDF)에 쓸 호텔 단위 본문 양 = 호텔별 본문 단어 수 합계")
display(hr.groupby("hotel_id")["text_words"].sum().describe(percentiles=[.1, .25, .5, .75]).to_frame("호텔별 본문 단어 수 합계").T)

## A-7. 호텔 지역 (전 세계 범위 확인)

In [ ]:
# 주의: hr에는 세부 평점 컬럼 'location'이 이미 있으므로 호텔 지역은 'region'으로 이름을 바꿔 합친다
loc = hr.merge(hotel_map[["hotel_id", "location"]].rename(columns={"location": "region"}), on="hotel_id")
top = loc.groupby("region").agg(리뷰수=("rating", "size"), 호텔수=("hotel_id", "nunique"),
                                  평균평점=("rating", "mean")).sort_values("리뷰수", ascending=False)
display(top.head(15))

t15 = top.head(15)[::-1]
plt.figure(figsize=(8, 5))
plt.barh(t15.index, t15["리뷰수"], color="#555")
plt.title("리뷰가 많은 지역 Top 15")
plt.xlabel("리뷰 수")
save("A7_top_locations")
print(f"지역 수: {top.shape[0]:,}개")

---
# B. Kaggle – 515K Hotel Reviews in Europe (테스트용)

Booking.com 리뷰이고 **사용자 ID가 없습니다**. 따라서 협업 필터링은 테스트할 수 없고,
콘텐츠 기반 모델과 여행자 유형(페르소나) 검증에 사용합니다. `rating`은 `reviewer_score_10 / 2` 입니다.

In [ ]:
kg = pd.read_csv(KAGGLE_DIR / "kaggle_test.csv", parse_dates=["review_date"])
meta = pd.read_csv(KAGGLE_DIR / "hotel_meta_kaggle.csv")
for c in ["positive_review", "negative_review", "review_text"]:
    kg[c] = kg[c].fillna("")
kg["pos_words"] = kg.positive_review.str.split().str.len()
kg["neg_words"] = kg.negative_review.str.split().str.len()
kg["text_words"] = kg.review_text.str.split().str.len()
print(f"불러오기 완료: {len(kg):,}행, 호텔 {kg.hotel_id.nunique():,}개")

## B-1. 기본 정보 (행 수, 컬럼, 데이터 타입, 결측, 중복)

In [ ]:
print(f"행 {len(kg):,} x 컬럼 {kg.shape[1]}, 호텔 {kg.hotel_id.nunique():,}개, 도시 {kg.city.nunique()}개")
print(f"기간: {kg.review_date.min().date()} ~ {kg.review_date.max().date()}")
display(kg.drop(columns=["pos_words", "neg_words", "text_words"]).dtypes.rename("dtype").to_frame().T)

m = kg.drop(columns=["pos_words", "neg_words", "text_words"]).isna().sum()
m = pd.DataFrame({"결측 수": m, "비율(%)": m / len(kg) * 100})
display(m[m["결측 수"] > 0])
print(f"본문이 완전히 빈 리뷰: {(kg.text_words == 0).sum():,}건 | 긍정 없음 {(kg.pos_words == 0).mean():.1%} | 부정 없음 {(kg.neg_words == 0).mean():.1%}")
print("완전 중복 행은 전처리에서 이미 제거됨 (전후 비교는 D절 참고)")
display(kg[["reviewer_score_10", "rating", "nights", "pos_words", "neg_words"]].describe().T)

## B-2. 평점 분포

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(kg.reviewer_score_10, bins=np.arange(2.25, 10.5, 0.5), color="#2ca02c")
ax[0].set_title("Reviewer_Score 분포 (10점 척도)")
ax[0].set_xlabel("점수")
ax[0].set_ylabel("리뷰 수")
r5 = kg.rating.round().astype(int).clip(1, 5).value_counts(normalize=True).reindex(range(1, 6), fill_value=0) * 100
ax[1].bar(r5.index, r5.values, color="#2ca02c")
ax[1].set_title("rating 분포 (5점 척도로 변환 후 반올림)")
ax[1].set_xlabel("rating")
ax[1].set_ylabel("비율 (%)")
save("B2_rating_distribution")
print(f"평균 {kg.rating.mean():.3f} (5점) / 중앙값 {kg.rating.median():.2f} / 4점 이상 비율 {(kg.rating >= 4).mean():.1%} / 왜도 {kg.rating.skew():.2f}")

## B-3. 도시·호텔

In [ ]:
city = kg.groupby("city").agg(리뷰수=("rating", "size"), 호텔수=("hotel_id", "nunique"), 평균평점=("rating", "mean"))
display(city.sort_values("리뷰수", ascending=False))

h = kg.groupby("hotel_id").size()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
c2 = city.sort_values("리뷰수")
ax[0].barh(c2.index, c2["리뷰수"], color="#2ca02c")
ax[0].set_title("도시별 리뷰 수")
ax[1].hist(h, bins=50, color="#2ca02c")
ax[1].set_title("호텔당 리뷰 수")
ax[1].set_xlabel("리뷰 수")
save("B3_city_hotel")
print(f"호텔당 리뷰 수: 최소 {h.min()}, 중앙값 {h.median():.0f}, 최대 {h.max()}")

## B-4. 여행 유형 (페르소나 검증용)

`Tags`에서 추출한 여행 목적(`trip_type`)과 여행자 유형(`traveler_type`)입니다.
HotelRec에서 도출한 여행자 유형을 이 컬럼으로 검증하려는 것이 제안서의 핵심 아이디어입니다.

In [ ]:
tt = kg.traveler_type.fillna("(없음)")
agg = kg.groupby(tt).agg(리뷰수=("rating", "size"), 평균평점=("rating", "mean"), 평균숙박일=("nights", "mean")).sort_values("리뷰수", ascending=False)
display(agg)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
a2 = agg.sort_values("리뷰수")
ax[0].barh(a2.index, a2["리뷰수"], color="#2ca02c")
ax[0].set_title("여행자 유형별 리뷰 수")
a3 = agg.sort_values("평균평점")
ax[1].barh(a3.index, a3["평균평점"], color="#2ca02c")
ax[1].set_xlim(max(0, a3["평균평점"].min() - 0.3), 5)
ax[1].set_title("여행자 유형별 평균 rating")
save("B4_traveler_type")

trip = kg.groupby(kg.trip_type.fillna("(없음)")).agg(리뷰수=("rating", "size"), 평균평점=("rating", "mean"))
display(trip)
print("여행 목적 x 여행자 유형 (행 기준 비율 %)")
display((pd.crosstab(kg.trip_type.fillna("(없음)"), tt, normalize="index") * 100))

In [ ]:
# 도시별 여행자 유형 구성과 평점
ct = pd.crosstab(kg.city, kg.traveler_type, normalize="index") * 100
display(ct)
pv = kg.pivot_table(index="city", columns="traveler_type", values="rating", aggfunc="mean")
display(pv)

## B-5. 숙박일수, 객실 유형, 모바일

In [ ]:
n = kg.nights.clip(upper=10).value_counts().sort_index()
by_n = kg.assign(nights_c=kg.nights.clip(upper=10)).groupby("nights_c")["rating"].mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(n.index.astype(int).astype(str), n.values, color="#2ca02c")
ax[0].set_title("숙박일수 (10박 초과는 10으로 표시)")
ax[0].set_xlabel("박")
ax[1].plot(by_n.index, by_n.values, marker="o", color="#2ca02c")
ax[1].set_title("숙박일수별 평균 rating")
ax[1].set_xlabel("박")
save("B5_nights")

display(kg.room_type.value_counts().head(10).to_frame("리뷰 수"))
print(f"모바일에서 작성: {kg.is_mobile.mean():.1%} | 반려동물 동반: {kg.with_pet.mean():.2%}")

## B-6. 시간 분포와 국적

In [ ]:
monthly = kg.set_index("review_date").resample("MS").size()
nat = kg.groupby("reviewer_nationality").agg(리뷰수=("rating", "size"), 평균평점=("rating", "mean")).sort_values("리뷰수", ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(monthly.index, monthly.values, marker="o", color="#2ca02c")
ax[0].set_title("월별 리뷰 수")
ax[0].tick_params(axis="x", rotation=30)
n15 = nat.head(15)[::-1]
ax[1].barh(n15.index, n15["리뷰수"], color="#2ca02c")
ax[1].set_title("리뷰어 국적 Top 15")
save("B6_time_nationality")
print(f"국적 수: {nat.shape[0]}개, 상위 3개국 비중 {nat.head(3)['리뷰수'].sum() / len(kg):.1%}")
display(nat.head(10))

## B-7. 리뷰 텍스트

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(kg.pos_words.clip(upper=100), bins=50, color="#2ca02c")
ax[0].set_title("긍정 리뷰 단어 수 (100 초과는 100)")
ax[1].hist(kg.neg_words.clip(upper=100), bins=50, color="#d62728")
ax[1].set_title("부정 리뷰 단어 수 (100 초과는 100)")
save("B7_text_length")
display(kg[["pos_words", "neg_words", "text_words"]].describe(percentiles=[.5, .9, .99]).T)

# 긍정/부정 작성 여부별 평균 점수
g = kg.assign(긍정=kg.pos_words > 0, 부정=kg.neg_words > 0).groupby(["긍정", "부정"])["rating"].agg(["size", "mean"])
g.columns = ["리뷰수", "평균rating"]
display(g)

## B-8. 호텔 단위 점수 일관성 (콘텐츠 기반 테스트의 정답 신호 점검)

In [ ]:
hm = kg.groupby("hotel_id")["rating"].mean().rename("mean_rating_5").to_frame().join(
    meta.set_index("hotel_id")[["average_score", "n_reviews_in_data"]])
hm["avg_score_5"] = hm["average_score"] / 2
r = hm[["mean_rating_5", "avg_score_5"]].corr().iloc[0, 1]

plt.figure(figsize=(5.5, 5))
plt.scatter(hm.avg_score_5, hm.mean_rating_5, s=10, alpha=0.5, color="#2ca02c")
lim = [min(hm.avg_score_5.min(), hm.mean_rating_5.min()) - 0.05, 5.02]
plt.plot(lim, lim, "k--", lw=1)
plt.xlabel("Booking 평균 점수 / 2 (average_score)")
plt.ylabel("데이터 내 리뷰 평균 rating")
plt.title(f"호텔 평균 점수 비교 (상관 {r:.3f})")
save("B8_hotel_avg_consistency")
print(f"호텔 {len(hm):,}개, 상관계수 {r:.3f}")

---
# C. 두 데이터셋 비교

두 데이터셋은 **플랫폼, 평점 척도, 기간, 리뷰 형식이 다릅니다.** 일반화 테스트 결과를 해석할 때 이 차이를 고려해야 합니다.
(Kaggle은 10점 척도를 5점으로 변환해 가장 가까운 정수로 맞춘 뒤 비교합니다.)

In [ ]:
hr_d = hr["rating"].round().astype(int).clip(1, 5).value_counts(normalize=True).reindex(range(1, 6), fill_value=0) * 100
kg_d = kg["rating"].round().astype(int).clip(1, 5).value_counts(normalize=True).reindex(range(1, 6), fill_value=0) * 100

x = np.arange(1, 6)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(x - 0.2, hr_d.values, 0.4, label="HotelRec", color=C_TRAIN)
ax[0].bar(x + 0.2, kg_d.values, 0.4, label="Kaggle", color="#2ca02c")
ax[0].set_title("평점 분포 비교 (5점 척도)")
ax[0].set_xlabel("rating")
ax[0].set_ylabel("비율 (%)")
ax[0].legend()

ax[1].hist(np.log10(hr.text_words.clip(lower=1)), bins=50, alpha=0.6, density=True, label="HotelRec 본문", color=C_TRAIN)
ax[1].hist(np.log10(kg.text_words.clip(lower=1)), bins=50, alpha=0.6, density=True, label="Kaggle 본문(긍정+부정)", color="#2ca02c")
ax[1].set_title("본문 길이 분포 (log10 단어 수)")
ax[1].legend()
save("C_comparison")

def _col(df, d, tcol):
    return [f"{len(df):,}", f"{df.rating.mean():.2f}", f"{df.rating.median():.2f}",
            f"{(df.rating >= 4).mean() * 100:.1f}", f"{df.text_words.median():.0f}",
            f"{d.min().year}~{d.max().year}"]

comp = pd.DataFrame({
    "HotelRec": _col(hr, hr.date, "date"),
    "Kaggle": _col(kg, kg.review_date, "review_date"),
}, index=["리뷰 수", "평균 rating(5점)", "중앙값", "4점 이상 비율(%)", "본문 단어 수 중앙값", "기간"])
display(comp)

---
# D. 전처리 전후 비교

전처리 스크립트가 저장한 `reports/*.csv`를 그대로 보여줍니다.

In [ ]:
pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 250
p1 = HOTELREC_DIR / "reports" / "hotelrec_report.csv"
p2 = KAGGLE_DIR / "reports" / "kaggle_report.csv"
if p1.exists():
    print("HotelRec 단계별 전후 비교")
    display(pd.read_csv(p1))
else:
    print("hotelrec_report.csv 없음:", p1)
if p2.exists():
    print("Kaggle 전처리 전후 비교")
    display(pd.read_csv(p2).set_index("stage").T)
else:
    print("kaggle_report.csv 없음:", p2)
pd.options.display.float_format = "{:,.3f}".format

---
# E. 핵심 수치와 해석 메모

In [ ]:
print("[HotelRec]")
print(f" - 리뷰 {len(hr):,}건, 사용자 {hr.user_id.nunique():,}명, 호텔 {hr.hotel_id.nunique():,}개, 기간 {hr.date.min().year}~{hr.date.max().year}")
print(f" - 평균 평점 {hr.rating.mean():.2f}, 4점 이상 {(hr.rating >= 4).mean():.1%}")
print(f" - 세부 평점 6개 완전 리뷰 {(hr.n_aspects == 6).mean():.1%} (train {len(train):,}건, 사용자 {train.user_id.nunique():,}명)")
print(f" - train 사용자 중 완전한 리뷰 5건 이상 {(u_train >= 5).sum():,}명, 7건 이상 {(u_train >= 7).sum():,}명")
print(f" - 사용자당 리뷰 수 중앙값 {u_all.median():.0f}, 호텔당 중앙값 {h_all.median():.0f}")
print(f" - 희소도 {1 - len(hr) / (hr.user_id.nunique() * hr.hotel_id.nunique()):.4%}")
print("[Kaggle]")
print(f" - 리뷰 {len(kg):,}건, 호텔 {kg.hotel_id.nunique():,}개, 6개 도시, 기간 {kg.review_date.min().date()}~{kg.review_date.max().date()}")
print(f" - 평균 rating {kg.rating.mean():.2f}, 4점 이상 {(kg.rating >= 4).mean():.1%}")
top_t = kg.traveler_type.value_counts(normalize=True)
print(f" - 가장 많은 여행자 유형: {top_t.index[0]} ({top_t.iloc[0]:.1%}), 가장 적은 유형: {top_t.index[-1]} ({top_t.iloc[-1]:.1%})")

## 해석 메모 (그래프를 확인한 뒤 팀에서 채워 넣으세요)

**HotelRec**
- 평점 분포의 특징(쏠림 여부, train과 partial의 차이):
- 세부 평점 중 결측이 가장 많은/적은 항목과 그 의미:
- 사용자당 리뷰 수 분포의 특징과, 사용자별 aspect 중요도 회귀 가능 범위:
- 시간에 따른 세부 평점 보유 비율 변화:

**Kaggle**
- 평점 분포의 특징(HotelRec과의 차이):
- 여행자 유형별 점수·숙박일수 차이:
- 도시/국적 쏠림:

**모델링에 주는 시사점**
- 평점 척도·분포 차이를 어떻게 보정할지:
- Kaggle에서 어떤 지표로 검증할지(사용자 ID가 없으므로):
- 추가 전처리가 필요한 항목: